# IndiaKart E-Commerce Analytics Project
## Phase 3 — KPI Calculations

**Name:** Kartik Jaswal

### Objective

The objective of this phase is to calculate key business performance indicators used by management to evaluate IndiaKart's revenue, customer behaviour, order performance, payments, returns, and inventory efficiency.

The KPIs will be calculated and validated using Pandas, SQL, and Excel to ensure consistency across analytical tools.

In [21]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [22]:
base_path = r"C:\Users\karti\Desktop\IndiaEKART\cleaned_data"

customers = pd.read_csv(os.path.join(base_path, "customers_clean.csv"))
orders = pd.read_csv(os.path.join(base_path, "orders_clean.csv"))
order_items = pd.read_csv(os.path.join(base_path, "order_items_clean.csv"))
products = pd.read_csv(os.path.join(base_path, "products_clean.csv"))
payments = pd.read_csv(os.path.join(base_path, "payments_clean.csv"))
returns = pd.read_csv(os.path.join(base_path, "returns_clean.csv"))
inventory = pd.read_csv(os.path.join(base_path, "inventory_clean.csv"))

print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Products:", products.shape)
print("Payments:", payments.shape)
print("Returns:", returns.shape)
print("Inventory:", inventory.shape)

C:\Users\karti\AppData\Local\Temp\ipykernel_12996\1151909058.py:4: DtypeWarning: Columns (19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  orders = pd.read_csv(os.path.join(base_path, "orders_clean.csv"))


Customers: (10000, 17)
Orders: (50000, 21)
Order Items: (100000, 11)
Products: (1000, 17)
Payments: (50000, 13)
Returns: (10000, 10)
Inventory: (1000, 11)


# KPI Calculations

## KPI 1 — Gross Merchandise Value (GMV)

**Definition:** GMV represents the total value of all orders placed on the IndiaKart platform, including delivered, cancelled, returned, shipped, and processing orders.

**Formula:**  
GMV = Sum of `final_amount` for all orders

In [23]:
# Validate final_amount before calculating GMV

print("Final Amount Data Type:")
print(orders['final_amount'].dtype)

print("\nMissing Values:")
print(orders['final_amount'].isna().sum())

print("\nNegative Values:")
print((orders['final_amount'] < 0).sum())

print("\nTotal Orders:")
print(orders['order_id'].nunique())

Final Amount Data Type:
float64

Missing Values:
0

Negative Values:
0

Total Orders:
50000


In [24]:
# Calculate Gross Merchandise Value (GMV)

gmv = orders['final_amount'].sum()

print("Gross Merchandise Value (GMV)")
print(f"₹{gmv:,.2f}")

Gross Merchandise Value (GMV)
₹3,147,385,544.58


## KPI 2 — Net Revenue

**Definition:** Net Revenue represents the total value of successfully delivered orders and reflects the actual revenue earned from completed orders.

**Formula:**  
Net Revenue = Sum of `final_amount` where `status = Delivered`

**Business Importance:**  
Net Revenue provides a more realistic measure of earned revenue because cancelled, returned, shipped, and processing orders are excluded.

In [25]:
# Validate order status before calculating Net Revenue

print("Order Status Distribution:")
print(orders['status'].value_counts())

print("\nMissing Status Values:")
print(orders['status'].isna().sum())

print("\nDelivered Orders:")
print((orders['status'] == 'Delivered').sum())

Order Status Distribution:
status
Delivered     32499
Cancelled      5996
Shipped        5071
Returned       3933
Processing     2501
Name: count, dtype: int64

Missing Status Values:
0

Delivered Orders:
32499


In [26]:
# Calculate Net Revenue from Delivered orders

delivered_orders = orders[
    orders['status'] == 'Delivered'
]

net_revenue = delivered_orders['final_amount'].sum()

print("Delivered Orders:", len(delivered_orders))
print(f"Net Revenue: ₹{net_revenue:,.2f}")

Delivered Orders: 32499
Net Revenue: ₹2,053,858,611.96


## KPI 3 — Average Order Value (AOV)

**Definition:** Average Order Value (AOV) measures the average revenue generated from each successfully delivered order.

**Formula:**

AOV = Net Revenue ÷ Number of Delivered Orders

**Business Importance:**

A higher AOV indicates that customers are purchasing higher-value products or more products per order, leading to increased revenue without increasing order volume.

In [27]:
# Calculate Average Order Value (AOV)

delivered_order_count = delivered_orders['order_id'].nunique()

aov = net_revenue / delivered_order_count

print("Delivered Orders:", delivered_order_count)
print(f"Net Revenue: ₹{net_revenue:,.2f}")
print(f"Average Order Value (AOV): ₹{aov:,.2f}")

Delivered Orders: 32499
Net Revenue: ₹2,053,858,611.96
Average Order Value (AOV): ₹63,197.59


### Observation

The Average Order Value (AOV) is ₹63,197.59, indicating that each successfully delivered order generates approximately ₹63.2 thousand in revenue on average.

A higher AOV reflects stronger customer spending per transaction and is an important indicator of revenue efficiency. Increasing AOV through cross-selling, upselling, and bundled offers can significantly improve overall revenue without increasing order volume.

## KPI 4 — Cancellation Rate

**Definition:** Cancellation Rate measures the percentage of orders that were cancelled before completion.

**Formula:**

Cancellation Rate = (Cancelled Orders ÷ Total Orders) × 100

**Business Importance:**

A high cancellation rate indicates operational inefficiencies, inventory shortages, payment issues, or customer order abandonment. Lower cancellation rates improve revenue realization and customer satisfaction.

**Target Benchmark:** Less than 10%

In [28]:
# KPI 4 - Cancellation Rate

total_orders = orders['order_id'].nunique()

cancelled_orders = (
    orders['status'] == 'Cancelled'
).sum()

cancellation_rate = (
    cancelled_orders / total_orders
) * 100

print("Total Orders:", total_orders)
print("Cancelled Orders:", cancelled_orders)
print(f"Cancellation Rate: {cancellation_rate:.2f}%")

Total Orders: 50000
Cancelled Orders: 5996
Cancellation Rate: 11.99%


### Observation

The cancellation rate is **11.99%**, which exceeds the business target of **10%**.

This indicates that approximately one out of every eight orders is cancelled before completion. Reducing cancellations through better inventory management, payment reliability, and operational improvements can significantly increase realized revenue.

## KPI 5 — Return Rate

**Definition:** Return Rate measures the percentage of delivered orders that were returned by customers.

**Formula:**

Return Rate = (Total Return Records ÷ Delivered Orders) × 100

**Business Importance:**

A high return rate may indicate product quality issues, inaccurate product descriptions, sizing problems, or logistics-related damage. Reducing returns improves profitability and customer satisfaction.

**Target Benchmark:** Less than 8%

In [29]:
# KPI 5 - Return Rate

total_returns = returns['return_id'].nunique()

delivered_orders = (
    orders['status'] == 'Delivered'
).sum()

return_rate = (
    total_returns / delivered_orders
) * 100

print("Delivered Orders:", delivered_orders)
print("Total Return Records:", total_returns)
print(f"Return Rate: {return_rate:.2f}%")

Delivered Orders: 32499
Total Return Records: 10000
Return Rate: 30.77%


### Observation

The return rate is 30.77%, which is significantly above the target benchmark of less than 8%. This indicates substantial exposure to product quality, sizing, fulfillment, or delivery-related issues.

Because the dataset contains 10,000 return records against 32,499 delivered orders, the result should be reported as calculated while also noting that the dataset may represent synthetic or broader return records than a typical operational dataset.

In [30]:
# Merge returns with order status

returns_validation = returns.merge(
    orders[['order_id', 'status']],
    on='order_id',
    how='left'
)

print(returns_validation['status'].value_counts())

status
Delivered    4327
Returned     3933
Cancelled    1067
Shipped       673
Name: count, dtype: int64


In [31]:
print("Return Records:", len(returns))

print("Unique Returned Orders:",
      returns['order_id'].nunique())

Return Records: 10000
Unique Returned Orders: 10000


In [32]:
returns_validation = returns.merge(
    orders[['order_id', 'status']],
    on='order_id',
    how='left'
)

print(returns_validation['status'].value_counts())

print("\nReturn Records:", len(returns))
print("Unique Returned Orders:", returns['order_id'].nunique())

status
Delivered    4327
Returned     3933
Cancelled    1067
Shipped       673
Name: count, dtype: int64

Return Records: 10000
Unique Returned Orders: 10000


In [33]:
# Valid returns linked only to Delivered orders

valid_returns = returns_validation[
    returns_validation['status'] == 'Delivered'
]

valid_return_count = valid_returns['return_id'].nunique()

valid_return_rate = (
    valid_return_count / delivered_orders
) * 100

print("Valid Delivered Returns:", valid_return_count)
print("Delivered Orders:", delivered_orders)
print(f"Valid Return Rate: {valid_return_rate:.2f}%")

Valid Delivered Returns: 4327
Delivered Orders: 32499
Valid Return Rate: 13.31%


### Data Validation & Business Logic Verification

Before accepting the Return Rate KPI, the relationship between **returns** and **order status** was validated.

#### Validation Results

| Order Status | Return Records |
|--------------|---------------:|
| Delivered | 4,327 |
| Returned | 3,933 |
| Cancelled | 1,067 |
| Shipped | 673 |

The validation revealed a **data quality issue**. According to standard e-commerce business rules, a product can only be returned after it has been successfully delivered. However, this dataset contains return records linked to **Cancelled** and **Shipped** orders.

Therefore, the raw Return Rate of **30.77%** does not accurately represent customer returns from completed deliveries.

#### Business-Correct Return Rate

- Valid Delivered Returns: **4,327**
- Delivered Orders: **32,499**
- **Validated Return Rate:** **13.31%**

### Final Business Conclusion

The **13.31%** value is the business-correct Return Rate because it considers only return records associated with delivered orders. The discrepancy highlights a data quality issue within the synthetic dataset and demonstrates the importance of validating business logic before reporting KPIs.

## KPI 6 — Customer Lifetime Value (CLV)

**Definition:**

Customer Lifetime Value (CLV) represents the average total amount spent by customers during their relationship with the business.

**Formula**

CLV = Average Total Spending per Customer Segment

**Business Importance**

CLV helps identify the most valuable customer segments and supports marketing, retention, and customer acquisition strategies.

**Business Benchmark**

Premium customers are expected to generate significantly higher lifetime value than Regular customers.

In [34]:
# KPI 6 - Customer Lifetime Value (CLV)

clv = (
    customers
    .groupby('segment')['total_spent']
    .agg(
        customer_count='count',
        average_clv='mean',
        total_revenue='sum'
    )
    .round(2)
    .sort_values('average_clv', ascending=False)
)

print(clv)

          customer_count  average_clv  total_revenue
segment                                             
Premium             1501    532735.00   7.996352e+08
Regular             4026    361870.72   1.456892e+09
New                 1488    274866.48   4.090013e+08
Budget              2505    186178.81   4.663779e+08
Inactive             480     32249.08   1.547956e+07


### Observation

The Customer Lifetime Value (CLV) analysis shows significant differences across customer segments.

- **Premium** customers have the highest average CLV of **₹532,735.00**, despite representing only **1,501 customers**. This indicates that Premium customers contribute the highest value per customer and should be the primary focus for retention and loyalty programs.
- **Regular** customers generate the highest total revenue (**₹1.46 billion**) because they represent the largest customer segment (**4,026 customers**), even though their average CLV (**₹361,870.72**) is lower than Premium customers.
- **New** and **Budget** customers have moderate lifetime values, suggesting an opportunity to increase their spending through personalized recommendations, cross-selling, and upselling strategies.
- **Inactive** customers have the lowest average CLV (**₹32,249.08**), indicating limited long-term value and highlighting the need for targeted reactivation campaigns.

### Business Validation

The project benchmark states that **Premium customers should generate approximately three times the lifetime value of Regular customers**.

**Validation Result:**

- Premium CLV = **₹532,735.00**
- Regular CLV = **₹361,870.72**
- Premium-to-Regular Ratio = **1.47×**

The dataset **does not satisfy the expected benchmark** of **3×**. Premium customers spend more than Regular customers, but only by approximately **47%**, indicating that the value gap between these segments is smaller than expected. This insight suggests an opportunity to strengthen Premium customer offerings or improve customer segmentation strategies.

## KPI 7 — Month-over-Month (MoM) Growth

### Definition

Month-over-Month (MoM) Growth measures the percentage change in Gross Merchandise Value (GMV) from one month to the next.

### Formula

MoM Growth (%) = ((Current Month GMV − Previous Month GMV) ÷ Previous Month GMV) × 100

### Business Importance

Month-over-Month Growth helps management monitor revenue trends over time. It highlights periods of business growth or decline, identifies seasonal demand patterns, and supports forecasting and strategic decision-making.

In [35]:
# Check order_date before MoM calculation

print("Order Date Data Type:")
print(orders['order_date'].dtype)

print("\nMissing Order Dates:")
print(orders['order_date'].isna().sum())

print("\nMinimum Date:")
print(orders['order_date'].min())

print("Maximum Date:")
print(orders['order_date'].max())

Order Date Data Type:
object

Missing Order Dates:
0

Minimum Date:
01-01-2024
Maximum Date:
31-12-2024


In [36]:
print(orders['order_date'].head(10))

print("\nLast 10 sample values:")
print(orders['order_date'].tail(10))

0    22-05-2025
1    16-10-2023
2    08-10-2023
3    02-10-2023
4    15-06-2024
5    02-03-2025
6    12-04-2025
7    12-08-2024
8    06-02-2024
9    07-08-2024
Name: order_date, dtype: object

Last 10 sample values:
49990    21-05-2024
49991    08-09-2024
49992    17-08-2023
49993    17-01-2024
49994    09-09-2023
49995    30-11-2023
49996    29-03-2025
49997    14-12-2024
49998    02-01-2025
49999    10-05-2024
Name: order_date, dtype: object


In [37]:
# Convert order_date to datetime

orders['order_date'] = pd.to_datetime(
    orders['order_date'],
    format='%d-%m-%Y',
    errors='coerce'
)

print("Order Date Data Type:")
print(orders['order_date'].dtype)

print("\nMinimum Date:")
print(orders['order_date'].min())

print("Maximum Date:")
print(orders['order_date'].max())

print("\nMissing Dates After Conversion:")
print(orders['order_date'].isna().sum())

Order Date Data Type:
datetime64[ns]

Minimum Date:
2023-06-24 00:00:00
Maximum Date:
2025-06-23 00:00:00

Missing Dates After Conversion:
0


In [38]:
# Create Year-Month column
orders['year_month'] = orders['order_date'].dt.to_period('M')

# Calculate monthly GMV using all order statuses
monthly_gmv = (
    orders
    .groupby('year_month', as_index=False)
    .agg(
        monthly_gmv=('final_amount', 'sum'),
        total_orders=('order_id', 'nunique')
    )
)

# Add GMV in millions for readability
monthly_gmv['gmv_million'] = (
    monthly_gmv['monthly_gmv'] / 1_000_000
).round(2)

print(monthly_gmv)

print("\nNumber of Calendar Months:", len(monthly_gmv))
print(f"Total GMV Check: ₹{monthly_gmv['monthly_gmv'].sum():,.2f}")

   year_month   monthly_gmv  total_orders  gmv_million
0     2023-06  2.738485e+07           444        27.38
1     2023-07  1.289205e+08          2081       128.92
2     2023-08  1.281220e+08          2091       128.12
3     2023-09  1.374625e+08          2104       137.46
4     2023-10  1.324709e+08          2060       132.47
5     2023-11  1.372634e+08          2114       137.26
6     2023-12  1.369234e+08          2090       136.92
7     2024-01  1.314500e+08          2090       131.45
8     2024-02  1.209459e+08          1907       120.95
9     2024-03  1.336701e+08          2111       133.67
10    2024-04  1.315097e+08          2131       131.51
11    2024-05  1.363584e+08          2167       136.36
12    2024-06  1.237410e+08          2072       123.74
13    2024-07  1.270283e+08          2025       127.03
14    2024-08  1.319235e+08          2146       131.92
15    2024-09  1.305253e+08          2011       130.53
16    2024-10  1.335183e+08          2131       133.52
17    2024

In [39]:
# Calculate Month-over-Month Growth

monthly_gmv['previous_month_gmv'] = (
    monthly_gmv['monthly_gmv'].shift(1)
)

monthly_gmv['mom_growth_pct'] = (
    (
        monthly_gmv['monthly_gmv']
        - monthly_gmv['previous_month_gmv']
    )
    / monthly_gmv['previous_month_gmv']
    * 100
).round(2)

print(monthly_gmv[
    [
        'year_month',
        'monthly_gmv',
        'previous_month_gmv',
        'mom_growth_pct'
    ]
])

   year_month   monthly_gmv  previous_month_gmv  mom_growth_pct
0     2023-06  2.738485e+07                 NaN             NaN
1     2023-07  1.289205e+08        2.738485e+07          370.77
2     2023-08  1.281220e+08        1.289205e+08           -0.62
3     2023-09  1.374625e+08        1.281220e+08            7.29
4     2023-10  1.324709e+08        1.374625e+08           -3.63
5     2023-11  1.372634e+08        1.324709e+08            3.62
6     2023-12  1.369234e+08        1.372634e+08           -0.25
7     2024-01  1.314500e+08        1.369234e+08           -4.00
8     2024-02  1.209459e+08        1.314500e+08           -7.99
9     2024-03  1.336701e+08        1.209459e+08           10.52
10    2024-04  1.315097e+08        1.336701e+08           -1.62
11    2024-05  1.363584e+08        1.315097e+08            3.69
12    2024-06  1.237410e+08        1.363584e+08           -9.25
13    2024-07  1.270283e+08        1.237410e+08            2.66
14    2024-08  1.319235e+08        1.270

### Observation

The Month-over-Month (MoM) analysis shows that monthly GMV remained relatively stable throughout most of the analysis period, with normal fluctuations between positive and negative growth.

The highest recorded growth (**370.77%**) occurred in **July 2023**. However, this should not be interpreted as actual business growth because **June 2023 contains only partial-month data (24–30 June)**.

Similarly, the largest decline (**-23.74%**) occurred in **June 2025**, which also represents a partial month and should therefore be excluded from business performance evaluation.

After excluding partial months, the strongest business growth occurred in **March 2025 (+19.61%)**, while the largest valid decline occurred in **February 2025 (-17.26%)**.

Overall, the business demonstrates relatively stable monthly revenue with moderate seasonal fluctuations rather than continuous growth or decline.

## KPI 8 — Top Category Revenue Share

### Definition

Top Category Revenue Share measures the percentage contribution of each product category to the total delivered revenue.

### Formula

Category Revenue Share (%) = (Category Revenue ÷ Total Revenue) × 100

### Business Importance

This KPI helps identify whether the business is overly dependent on a single product category. High concentration in one category increases business risk, while a diversified revenue mix provides greater stability.

### Business Benchmark

A single category contributing more than **60%** of total revenue indicates a potential concentration risk.

In [40]:
print("Category Missing Values:")
print(order_items['category'].isna().sum())

print("\nTotal Price Missing Values:")
print(order_items['total_price'].isna().sum())

print("\nUnique Categories:")
print(order_items['category'].nunique())

print("\nCategories:")
print(order_items['category'].value_counts())

Category Missing Values:
0

Total Price Missing Values:
0

Unique Categories:
10

Categories:
category
Office Supplies     10152
Home & Kitchen      10109
Electronics         10090
Grocery             10026
Fashion             10002
Toys & Baby          9989
Automotive           9986
Books                9949
Sports & Fitness     9903
Beauty & Health      9794
Name: count, dtype: int64


In [41]:
delivered_items = order_items.merge(
    orders[['order_id', 'status']],
    on='order_id',
    how='left'
)

delivered_items = delivered_items[
    delivered_items['status'] == 'Delivered'
]

print("Rows After Merge:", len(delivered_items))
print("\nMissing Status:", delivered_items['status'].isna().sum())

Rows After Merge: 64951

Missing Status: 0


In [42]:
category_kpi = (
    delivered_items
    .groupby('category')['total_price']
    .sum()
    .reset_index()
)

category_kpi.columns = [
    'category',
    'category_revenue'
]

total_revenue = category_kpi['category_revenue'].sum()

category_kpi['revenue_share_pct'] = (
    category_kpi['category_revenue']
    / total_revenue
    * 100
).round(2)

category_kpi = category_kpi.sort_values(
    'revenue_share_pct',
    ascending=False
)

category_kpi

,category,category_revenue,revenue_share_pct
3,Electronics,1.076374e+09,57.74
8,Sports & Fitness,3.192419e+08,17.13
6,Home & Kitchen,1.576279e+08,8.46
4,Fashion,9.691033e+07,5.20
0,Automotive,8.063293e+07,4.33
7,Office Supplies,6.070733e+07,3.26
9,Toys & Baby,2.990482e+07,1.60
1,Beauty & Health,2.168397e+07,1.16
5,Grocery,1.207455e+07,0.65
2,Books,9.011382e+06,0.48


### Observation

The category revenue analysis shows that **Electronics** is the largest revenue contributor, generating **57.74%** of the total delivered revenue.

Although Electronics dominates overall sales, its contribution remains **below the 60% concentration risk benchmark**, indicating that the business is **not excessively dependent on a single product category**.

The remaining revenue is distributed across multiple categories, with **Sports & Fitness (17.13%)** and **Home & Kitchen (8.46%)** serving as important secondary revenue contributors.

Overall, the revenue distribution suggests a relatively diversified product portfolio while highlighting Electronics as the primary growth driver.

### Business Validation

**Benchmark:** A single product category contributing more than **60%** of total revenue indicates concentration risk.

**Result**

- Electronics Revenue Share = **57.74%**
- Risk Threshold = **60.00%**

**Conclusion**

The business **does not exceed** the concentration risk threshold. Although Electronics is the dominant revenue category, the current revenue mix remains reasonably diversified.

## KPI 9 — Payment Failure Rate

### Definition

Payment Failure Rate measures the percentage of payment transactions that failed during the payment process.

### Formula

Payment Failure Rate = (Failed Payments ÷ Total Payments) × 100

### Business Importance

A high payment failure rate can lead to lost revenue, poor customer experience, and lower order conversion rates. Monitoring this KPI helps identify payment gateway issues and improve transaction success rates.

### Business Benchmark

Target Payment Failure Rate: **Less than 2%**

In [45]:
payments.columns.tolist()


['payment_id',
 'order_id',
 'customer_id',
 'payment_date',
 'payment_time',
 'payment_method',
 'amount',
 'status',
 'transaction_id',
 'bank_name',
 'gateway',
 'refund_amount',
 'refund_date']

In [46]:
print("Payment Status Data Type:")
print(payments['status'].dtype)

print("\nMissing Payment Status:")
print(payments['status'].isna().sum())

print("\nPayment Status Distribution:")
print(payments['status'].value_counts())

print("\nTotal Payments:")
print(len(payments))

Payment Status Data Type:
object

Missing Payment Status:
0

Payment Status Distribution:
status
Success    48252
Failed      1748
Name: count, dtype: int64

Total Payments:
50000


In [47]:
# KPI 9 - Payment Failure Rate

total_payments = len(payments)

failed_payments = (
    payments['status'] == 'Failed'
).sum()

payment_failure_rate = (
    failed_payments / total_payments
) * 100

print("Total Payments:", total_payments)
print("Failed Payments:", failed_payments)
print(f"Payment Failure Rate: {payment_failure_rate:.2f}%")

Total Payments: 50000
Failed Payments: 1748
Payment Failure Rate: 3.50%


### Observation

The platform processed **50,000 payment transactions**, of which **1,748 failed**, resulting in a **Payment Failure Rate of 3.50%**.

This exceeds the business benchmark of **2.00%**, indicating potential revenue leakage and customer friction during the payment process.

Improving payment gateway reliability, reducing transaction failures, and monitoring payment success rates can help increase completed purchases and overall revenue.

## KPI 10 — Inventory Fill Rate

### Definition

Inventory Fill Rate measures the percentage of products that are currently available for sale.

### Formula

Inventory Fill Rate = (In-Stock SKUs ÷ Total SKUs) × 100

### Business Importance

A high inventory fill rate ensures that customers can purchase products without stock shortages. A low fill rate may lead to missed sales opportunities and reduced customer satisfaction.

### Business Benchmark

Target Inventory Fill Rate: **Greater than 95%**

In [51]:
inventory.columns.tolist()

['inventory_id',
 'product_id',
 'warehouse_location',
 'quantity_available',
 'quantity_reserved',
 'reorder_level',
 'reorder_quantity',
 'last_restocked_date',
 'unit_cost',
 'total_inventory_value',
 'status']

In [53]:
print("Inventory Shape:")
print(inventory.shape)

print("\nMissing Status:")
print(inventory['status'].isna().sum())

print("\nInventory Status Distribution:")
print(inventory['status'].value_counts())

print("\nTotal SKUs:")
print(len(inventory))


Inventory Shape:
(1000, 11)

Missing Status:
0

Inventory Status Distribution:
status
In Stock        922
Low Stock        76
Out of Stock      2
Name: count, dtype: int64

Total SKUs:
1000


In [54]:
# KPI 10 - Inventory Fill Rate

total_skus = len(inventory)

in_stock_skus = (
    inventory['status'] == 'In Stock'
).sum()

inventory_fill_rate = (
    in_stock_skus / total_skus
) * 100

print("Total SKUs:", total_skus)
print("In Stock SKUs:", in_stock_skus)
print(f"Inventory Fill Rate: {inventory_fill_rate:.2f}%")

Total SKUs: 1000
In Stock SKUs: 922
Inventory Fill Rate: 92.20%


### Observation

The inventory consists of **1,000 SKUs**, of which **922 are currently In Stock**, resulting in an **Inventory Fill Rate of 92.20%**.

This is **below the business benchmark of 95%**, indicating that product availability could be improved to reduce stock-related sales losses.

Although only **2 products are completely Out of Stock**, an additional **76 products are classified as Low Stock**, suggesting a need for timely replenishment to maintain healthy inventory levels.